__LLM Judge__

In [ ]:
import os, sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.cross_audit.llm_judge import run_llm_judge

# Read existed output_csv
# Skip [89/90] success，only rerun failed records
df_res = run_llm_judge(
    blind_csv="data/annotations/pilot_n15/pilot_n15_blind.csv",
    output_csv="data/annotations/pilot_n15/pilot_n15_llm_judge.csv",
    judge_key="claude_sonnet55",
)
print("最终成功数:", (df_res["judge_status"] == "success").sum(), "/", len(df_res))

05:24:45 | INFO    | Resuming from checkpoint: 89/90 already judged.
05:24:45 | INFO    | [21/90] Judging BLIND_021 (Task2_AIEthics) with claude-sonnet-5-5...
05:25:03 | INFO    | LLM-as-a-Judge completed -> data\annotations\pilot_n15\pilot_n15_llm_judge.csv (Success: 90/90)
最终成功数: 90 / 90


In [2]:
import os, sys
import pandas as pd
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 1. 先确认输入盲标表确实是完整的 90 条
blind_df = pd.read_csv("data/annotations/pilot_n15/pilot_n15_blind.csv")
print("输入盲标表总行数:", len(blind_df))  # 应打印 90

# 2. 继续执行全量审计（会自动跳过已完成的 BLIND_001 ~ BLIND_021，从 BLIND_022 开始补全）
from src.cross_audit.llm_judge import run_llm_judge

df_full = run_llm_judge(
    blind_csv="data/annotations/pilot_n15/pilot_n15_blind.csv",
    output_csv="data/annotations/pilot_n15/pilot_n15_llm_judge.csv",
    judge_key="claude_sonnet55",
)
print("完成后总行数:", len(df_full), "| 成功数:", (df_full["judge_status"] == "success").sum())

输入盲标表总行数: 90
05:34:55 | INFO    | Resuming from checkpoint: 21/90 already judged.
05:34:55 | INFO    | [22/90] Judging BLIND_022 (Task2_AIEthics) with claude-sonnet-5-5...
05:35:11 | INFO    | [23/90] Judging BLIND_023 (Task2_AIEthics) with claude-sonnet-5-5...
05:35:25 | INFO    | [24/90] Judging BLIND_024 (Task1_KNN) with claude-sonnet-5-5...
05:35:38 | INFO    | [25/90] Judging BLIND_025 (Task1_KNN) with claude-sonnet-5-5...
05:35:47 | INFO    | [26/90] Judging BLIND_026 (Task1_KNN) with claude-sonnet-5-5...
05:35:58 | INFO    | [27/90] Judging BLIND_027 (Task1_KNN) with claude-sonnet-5-5...
05:36:10 | INFO    | [28/90] Judging BLIND_028 (Task1_KNN) with claude-sonnet-5-5...
05:36:23 | INFO    | [29/90] Judging BLIND_029 (Task1_KNN) with claude-sonnet-5-5...
05:36:34 | INFO    | [30/90] Judging BLIND_030 (Task1_KNN) with claude-sonnet-5-5...
05:36:48 | INFO    | [31/90] Judging BLIND_031 (Task1_KNN) with claude-sonnet-5-5...
05:37:04 | INFO    | [32/90] Judging BLIND_032 (Task1_KNN)

In [3]:
import json, os, shutil, sys, time, yaml
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

# 1. 挂载项目根目录
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [5]:
import src.cross_audit.llm_judge as lj
load_dotenv(override=True)

out_csv = Path("data/annotations/pilot_n15/pilot_n15_llm_judge.csv")

# 2. 读取现有 CSV，仅锁定非 success 的剩余行索引
df = pd.read_csv(out_csv, dtype=str).fillna("")
failed_mask = df["judge_status"] != "success"
failed_indices = df[failed_mask].index.tolist()

print(f"当前已成功: {(~failed_mask).sum()} 条（原样锁定） | 本次仅重跑: {len(failed_indices)} 条")
print("待重跑 ID:", df.loc[failed_indices, "blind_id"].tolist())

# 3. 加载配置与 Codebook
with open("configs/judge_config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

judge_cfg = cfg["judges"]["claude_sonnet55"]
api_key = os.getenv(judge_cfg["api_key_env"]).strip()
sys_prompt = lj.build_system_prompt(Path(cfg["judge_settings"]["codebook_path"]).read_text(encoding="utf-8"))

# 4. 仅针对剩余索引逐条打分，统一转 str 写入避免 pandas dtype='str' 报错
for step, idx in enumerate(failed_indices, 1):
    row = df.loc[idx]
    blind_id, task_id = row["blind_id"], row["task_id"]
    user_prompt = lj.build_user_prompt(blind_id, task_id, row["raw_output"])

    for attempt in range(1, 4):
        try:
            raw_resp = lj.call_judge_api(
                provider=judge_cfg["provider"],
                model_id=judge_cfg["model_id"],
                api_key=api_key,
                params={"max_tokens": 4096},
                system_prompt=sys_prompt,
                user_prompt=user_prompt,
            )
            payload = lj.extract_json_payload(raw_resp)

            if task_id == "Task1_KNN":
                p = lj.Task1AuditSchema.model_validate(payload)
                pass_bin = int(all([
                    p.t1_c1_distance,
                    p.t1_c2_k_tradeoff,
                    p.t1_c3_decision_rule,
                    p.t1_c4_nature_no_hallucination,
                ]))
                vals = [
                    p.t1_c1_distance, p.t1_c2_k_tradeoff, p.t1_c3_decision_rule,
                    p.t1_c4_nature_no_hallucination, pass_bin,
                    p.t1_exemplar_evidence, p.t1_exemplar_domain,
                    p.coder_notes, judge_cfg["model_id"], "success"
                ]
                df.loc[idx, [
                    "t1_c1_distance", "t1_c2_k_tradeoff", "t1_c3_decision_rule",
                    "t1_c4_nature_no_hallucination", "t1_pass_binary",
                    "t1_exemplar_evidence", "t1_exemplar_domain",
                    "coder_notes", "judge_model", "judge_status"
                ]] = [str(v) for v in vals]
            else:
                p = lj.Task2AuditSchema.model_validate(payload)
                k_i = len(p.units)
                c1 = sum(u.code for u in p.units)
                vals = [
                    k_i, p.t2_quality_score,
                    json.dumps([u.model_dump() for u in p.units], ensure_ascii=False),
                    c1, round(c1 / k_i, 4) if k_i > 0 else 0.0,
                    p.coder_notes, judge_cfg["model_id"], "success"
                ]
                df.loc[idx, [
                    "t2_unit_count_k", "t2_quality_score", "t2_unit_breakdown_json",
                    "t2_code1_count", "t2_code1_ratio",
                    "coder_notes", "judge_model", "judge_status"
                ]] = [str(v) for v in vals]

            print(f"[{step}/{len(failed_indices)}] {blind_id} ({task_id}) -> OK")
            break
        except Exception as e:
            print(f"[{step}/{len(failed_indices)}] {blind_id} attempt {attempt} error: {e}")
            time.sleep(3 * attempt)

    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    time.sleep(0.5)

# 5. 最终核验
check_df = pd.read_csv(out_csv)
print(f"\n补跑完成！总行数: {len(check_df)} | 成功数: {(check_df['judge_status'] == 'success').sum()}")

当前已成功: 85 条（原样锁定） | 本次仅重跑: 5 条
待重跑 ID: ['BLIND_069', 'BLIND_070', 'BLIND_071', 'BLIND_072', 'BLIND_073']
[1/5] BLIND_069 (Task2_AIEthics) -> OK
[2/5] BLIND_070 (Task2_AIEthics) -> OK
[3/5] BLIND_071 (Task2_AIEthics) -> OK
[4/5] BLIND_072 (Task2_AIEthics) -> OK
[5/5] BLIND_073 (Task1_KNN) -> OK

补跑完成！总行数: 90 | 成功数: 90
